In [35]:
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
    roc_curve,
    roc_auc_score
)

In [36]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [37]:
payrolls = pd.read_csv("mlb_payrolls_2026.csv")
mlb_40man = pd.read_csv("mlb_40man_acquisition.csv")

In [ ]:
import re
import time
import pandas as pd
import requests
from bs4 import BeautifulSoup

from data import df as playoff_teams

BR_CODES = {"TBD": "TBR", "KC": "KCR", "WSH": "WSN"}

headers = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/115.0 Safari/537.36"
    ),
    "Accept-Language": "en-US,en;q=0.9",
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
    "Referer": "https://google.com",
}

player_rows = []

for _, row in playoff_teams.iterrows():
    year = row["Year"]
    team = BR_CODES.get(row["NAME"], row["NAME"])
    url = f"https://www.baseball-reference.com/teams/{team}/{year}.shtml"
    response = requests.get(url, headers=headers)
    response.encoding = "utf-8"

    if response.status_code != 200:
        print(f"Failed to fetch {team} {year}: Status {response.status_code}")
        time.sleep(4)
        continue

    soup = BeautifulSoup(response.text, "html.parser")
    meta = re.sub(r"\s+", " ", soup.find(id="meta").get_text(" ", strip=True))

    record = re.search(r"Record:\s*(\d+)-(\d+),\s*(?:Finished\s*)?(\w+?)(?:\s*place)?\s*in\s*(\w+)", meta)
    wins, losses, place, division = (
        (int(record.group(1)), int(record.group(2)), record.group(3), record.group(4))
        if record else (None, None, None, None)
    )
    post = re.search(r"Postseason:\s*(.*?)\s*Manager:", meta)
    results = re.split(r"\s(?=(?:Won|Lost) )", re.sub(r"\s+\(", " (", post.group(1))) if post else []

    players = {}
    for table_id in ["players_standard_batting", "players_standard_pitching"]:
        table = soup.find("table", id=table_id)
        if not table:
            continue
        for tr in table.find("tbody").find_all("tr"):
            name = tr.find(attrs={"data-stat": "name_display"})
            if name is None or "thead" in (tr.get("class") or []):
                continue
            pos = tr.find(attrs={"data-stat": "team_position"})
            age = tr.find(attrs={"data-stat": "age"})
            players.setdefault(name.get_text(strip=True).rstrip("*#"), {
                "Position": pos.get_text(strip=True) if pos else None,
                "Age": age.get_text(strip=True) if age else None,
            })

    for name, info in players.items():
        player_rows.append({
            "Name": name, **info, "Team": team, "Year": year,
            "Wins": wins, "Losses": losses, "Place": place, "Division": division,
            "Playoff Result": results[0] if results else None,
            "Postseason": "; ".join(results) if results else None,
        })

    print(team, year, wins, results[0] if results else None, len(players))
    time.sleep(3)

team_rosters = pd.DataFrame(player_rows)
team_rosters["Age"] = pd.to_numeric(team_rosters["Age"], errors="coerce")



CHW 2005 99 Won World Series (4-0) over Houston Astros 38
LAA 2005 95 Lost AL Championship Series (4-1) to Chicago White Sox 40
NYY 2005 95 Lost AL Division Series (3-2) to Los Angeles Angels of Anaheim 51
BOS 2005 95 Lost AL Division Series (3-0) to Chicago White Sox 52
STL 2005 100 Lost NL Championship Series (4-2) to Houston Astros 40
ATL 2005 90 Lost NL Division Series (3-1) to Houston Astros 45
SDP 2005 82 Lost NL Division Series (3-0) to St. Louis Cardinals 46
HOU 2005 None Lost World Series (4-0) to Chicago White Sox 36
NYY 2006 97 Lost AL Division Series (3-1) to Detroit Tigers 49
MIN 2006 96 Lost AL Division Series (3-0) to Oakland Athletics 39
OAK 2006 93 Lost AL Championship Series (4-0) to Detroit Tigers 40
DET 2006 95 Lost World Series (4-1) to St. Louis Cardinals 40
NYM 2006 97 Lost NL Championship Series (4-3) to St. Louis Cardinals 49
STL 2006 83 Won World Series (4-1) over Detroit Tigers 41
LAD 2006 88 Lost NL Division Series (3-0) to New York Mets 50
SDP 2006 88 Lost 

In [ ]:
import io
import pandas as pd
import requests

SHEET_ID = "12XSXOQpjDJDCJKsA4xC1e_9FlS11aeioZy_p1nqpclg"

SALARY_GIDS = {
    2005: 1355703338,
    2006: 570423787,
    2007: 1452438542,
    2008: 1037903398,
    2009: 43829577,
    2010: 1438932004,
    2011: 887731618,
    2012: 1702409228,
    2013: 627550772,
    2014: 2062534909,
    2015: 1369722664,
    2016: 1187664277,
    2017: 1656350490,
    2018: 1775573725,
    2019: 1932759411,
    2020: 285575536,
    2021: 1633405383,
    2022: 1811400237,
    2023: 1677021452,
    2024: 1937251654,
    2025: 387264548,
    2026: 162500738,
}

salary_dfs = {}

for year, gid in SALARY_GIDS.items():
    url = f"https://docs.google.com/spreadsheets/d/{SHEET_ID}/export?format=csv&gid={gid}"
    response = requests.get(url)

    if response.status_code != 200:
        print(f"Failed to fetch {year}: Status {response.status_code}")
        continue

    df = pd.read_csv(io.StringIO(response.text), header=1).iloc[:, :4]
    if df.shape[1] == 3:  # 2005-2009 tabs have no service time column
        df.insert(2, "MLS", float("nan"))
    df.columns = ["Name", "Pos", "MLS", "Salary"]
    df["Year"] = year
    salary_dfs[year] = df

salaries_all = pd.concat(salary_dfs.values(), ignore_index=True)

salaries_all["Name"] = salaries_all["Name"].astype(str).str.replace("*", "", regex=False).str.strip()
salaries_all["Salary"] = (
    salaries_all["Salary"]
    .astype(str)
    .str.replace(r"[^\d.-]", "", regex=True)
    .pipe(pd.to_numeric, errors="coerce")
)
salaries_all = salaries_all[salaries_all["Name"].str.upper() != "TOTAL"].dropna(subset=["Salary"])
salaries_all["Salary"] = salaries_all["Salary"].round().astype("int64")
salaries_all["MLS"] = pd.to_numeric(salaries_all["MLS"], errors="coerce")

salaries_all.groupby("Year")["Salary"].agg(["count", "max", "sum"])

In [6]:
from parse import standardize_names
from parse import parse_postseason

salaries_all_clean = standardize_names(salaries_all, "Name")

playoff_salary_df = pd.merge(
    team_rosters,
    salaries_all_clean,
    left_on=['Name', 'Year'],
    right_on=['Name', 'Year'],
    how='left'
)

playoff_salary_df.drop(columns=["Playoff Result", "Pos"], inplace=True)
playoff_salary_df["Round"] = playoff_salary_df["Postseason"].map(parse_postseason)


In [23]:
playoff_salary_df["aboveYearMedian"] = playoff_salary_df["Salary"] > playoff_salary_df.groupby("Year")["Salary"].transform("median")

salary_features_df = (
    playoff_salary_df.groupby(["Team", "Year"])
    .agg(
        Round=('Round', 'first'), # keep first value
        Wins=('Wins', 'first'),
        totalSalary=('Salary', 'sum'),
        avgSalary=('Salary', 'mean'),
        maxSalary=('Salary', 'max'),
        medianSalary=('Salary', 'median'),
        stdSalary=('Salary', 'std'),
        avgService=('MLS', 'mean'),
        rosterSize=('Name', 'count'),
        salaryMatched=('Salary', 'count'),
        gtMedianSalary=('aboveYearMedian', 'mean'),
        top3SalaryShare=('Salary', lambda x: x.nlargest(3).sum() / x.sum() if x.sum() > 0 else 0)
    )
    .reset_index()
)

salary_features_df["salaryMatchRate"] = salary_features_df["salaryMatched"] / salary_features_df["rosterSize"]

# salaries inflate over time, so also express them relative to the other playoff teams that year
for col in ["totalSalary", "avgSalary", "maxSalary", "medianSalary"]:
    salary_features_df[f"{col}Rel"] = salary_features_df[col] / salary_features_df.groupby("Year")[col].transform("mean")
salary_features_df["payrollRank"] = salary_features_df.groupby("Year")["totalSalary"].rank(pct=True)

In [15]:
predictors_df = salary_features_df.sort_values(["Team", "Year"]).reset_index(drop=True)

# only carry over last season's salary when the team also made the playoffs the year before
consecutive = predictors_df["Year"] - predictors_df.groupby("Team")["Year"].shift(1) == 1
predictors_df["prevTotalSalary"] = predictors_df.groupby("Team")["totalSalary"].shift(1).where(consecutive)
predictors_df["prevAvgSalary"] = predictors_df.groupby("Team")["avgSalary"].shift(1).where(consecutive)
predictors_df["salaryGrowthRate"] = (
    (predictors_df["totalSalary"] - predictors_df["prevTotalSalary"])
    / predictors_df["prevTotalSalary"]
)

# a prior-year total of 0 (no salaries matched) divides to inf, so treat it as missing
predictors_df = predictors_df.replace([np.inf, -np.inf], np.nan)

In [30]:
from scipy.stats import randint
from sklearn.base import clone
from sklearn.dummy import DummyClassifier
from sklearn.metrics import average_precision_score
from sklearn.model_selection import RandomizedSearchCV, RepeatedStratifiedKFold, cross_val_score

predictors_df["Champ"] = predictors_df["Round"] == 5

validation = predictors_df[predictors_df["Year"] == 2026]  # GOAL: predict 2026 playoff outcomes
filt_pred_df = predictors_df[(predictors_df["Year"] != 2026) & (predictors_df["Round"] > 0)]

FEATURE_DROP = ["Team", "Year", "Wins", "Round", "prevAvgSalary", "prevTotalSalary",
                "totalSalary", "avgSalary", "maxSalary", "medianSalary", "Champ"]
X = filt_pred_df.drop(columns=FEATURE_DROP)
y = filt_pred_df["Champ"]


X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)


pipe = make_pipeline(
    SimpleImputer(strategy="median", add_indicator=True),
    RandomForestClassifier(random_state=42, n_jobs=-1),
)

param_dist = {
    "randomforestclassifier__n_estimators": [300, 500, 800],
    "randomforestclassifier__max_depth": [3, 5, 8, None],
    "randomforestclassifier__min_samples_leaf": randint(1, 8),
    "randomforestclassifier__max_features": ["sqrt", "log2", 0.5],
    "randomforestclassifier__class_weight": [None, "balanced", "balanced_subsample"],
    "randomforestclassifier__max_samples": [None, 0.7, 0.9],
}

# folds can't exceed the size of the rarest class
n_splits = int(min(5, y_train.value_counts().min()))

cv = RepeatedStratifiedKFold(n_splits=n_splits, n_repeats=3, random_state=42)

# champions are rare, so score how well the probabilities rank them rather than hard labels
search = RandomizedSearchCV(
    pipe, param_dist, n_iter=40, cv=cv, scoring="average_precision", n_jobs=-1, random_state=42, verbose=1
)
search.fit(X_train, y_train)

print(search.best_params_)
print(search.best_score_)

best_model = search.best_estimator_

Fitting 15 folds for each of 40 candidates, totalling 600 fits
{'randomforestclassifier__class_weight': None, 'randomforestclassifier__max_depth': 3, 'randomforestclassifier__max_features': 0.5, 'randomforestclassifier__max_samples': None, 'randomforestclassifier__min_samples_leaf': 1, 'randomforestclassifier__n_estimators': 300}
0.23549150969438193


In [31]:
baseline = cross_val_score(DummyClassifier(strategy="prior"), X_train, y_train, cv=cv, scoring="average_precision")
print(f"baseline average_precision: {baseline.mean():.3f}")

test_proba = best_model.predict_proba(X_test)[:, 1]
print(f"test average_precision: {average_precision_score(y_test, test_proba):.3f}, test roc_auc: {roc_auc_score(y_test, test_proba):.3f}")

# only 25 seasons of data
# leave one season out: where does the real champion rank among that year's playoff teams?
champ_ranks = {}
for year in sorted(filt_pred_df["Year"].unique()):
    train, test = filt_pred_df[filt_pred_df["Year"] != year], filt_pred_df[filt_pred_df["Year"] == year]
    if not test["Champ"].any():
        continue
    
    model = clone(best_model).fit(train.drop(columns=FEATURE_DROP), train["Champ"])
    proba = pd.Series(model.predict_proba(test.drop(columns=FEATURE_DROP))[:, 1], index=test.index)
    champ_ranks[year] = int(proba.rank(ascending=False, method="min")[test["Champ"]].iloc[0])

champ_ranks = pd.Series(champ_ranks, name="champRank")
print(f"mean champion rank: {champ_ranks.mean():.1f}, top-3 in {(champ_ranks <= 3).sum()} of {len(champ_ranks)} seasons")

names = best_model[:-1].get_feature_names_out()
importances = pd.Series(best_model[-1].feature_importances_, names).sort_values(ascending=False)
importances.head(10)

baseline average_precision: 0.104
test average_precision: 0.146, test roc_auc: 0.632
mean champion rank: 3.7, top-3 in 10 of 21 seasons


rosterSize         0.172682
maxSalaryRel       0.143220
avgService         0.111170
salaryMatchRate    0.087101
stdSalary          0.086807
totalSalaryRel     0.065335
medianSalaryRel    0.062893
gtMedianSalary     0.059955
avgSalaryRel       0.049127
payrollRank        0.041256
dtype: float64

,Name,Position,Age,Team,Year,Wins,Losses,Place,Division,Postseason,MLS,Salary,Round,aboveYearMedian
0,A.J. Pierzynski,C,28,CHW,2005,99.0,63.0,1st,AL_Central,Won World Series (4-0) over Houston Astros; Wo...,NaN,NaN,5,False
1,Paul Konerko,1B,29,CHW,2005,99.0,63.0,1st,AL_Central,Won World Series (4-0) over Houston Astros; Wo...,NaN,NaN,5,False
2,Tadahito Iguchi,2B,30,CHW,2005,99.0,63.0,1st,AL_Central,Won World Series (4-0) over Houston Astros; Wo...,NaN,NaN,5,False
3,Juan Uribe,SS,26,CHW,2005,99.0,63.0,1st,AL_Central,Won World Series (4-0) over Houston Astros; Wo...,NaN,NaN,5,False
4,Joe Crede,3B,27,CHW,2005,99.0,63.0,1st,AL_Central,Won World Series (4-0) over Houston Astros; Wo...,NaN,NaN,5,False


In [32]:
X_val = validation.drop(columns=FEATURE_DROP)
val_proba = best_model.predict_proba(X_val)
class_idx = list(best_model.classes_).index(True)

title_odds = pd.DataFrame({
    "Team": validation["Team"].values,
    "ChampOdds": val_proba[:, class_idx],
}).sort_values("ChampOdds", ascending=False)

title_odds["Share"] = title_odds["ChampOdds"] / title_odds["ChampOdds"].sum()
title_odds

,Team,ChampOdds,Share
8,NYY,0.371637,0.291894
6,LAD,0.274842,0.215868
10,SDP,0.188586,0.148121
9,PHI,0.146658,0.115189
2,CHC,0.064456,0.050626
0,ATL,0.062008,0.048703
5,HOU,0.060302,0.047363
1,BOS,0.042596,0.033456
4,CLE,0.021685,0.017032
11,TBR,0.014637,0.011496
